# Fine-Tuning a Large Language Model for Layperson Medical Question Answering
AI in Healthcare | MSAI Spr '25 | Chelsea Ramos

## Objectives
1. **Generate ~3.5k medical multiple-choice questions (MCQs)**
    - **Using the MedQuAD dataset and Phi-2**
2. Fine-tune TinyLLaMA with the generated MCQs
3. Evaluate model accuracy on held-out MCQs
4. Compare against other models (baseline and zero-shot)

---

# 1. Generate Data

In [2]:
import pandas as pd

## 1.1 Load & Clean Data

- Dataset: MedQuAD 
    - 47,457 layperson medical question-answer pairs

In [3]:
df = pd.read_parquet("hf://datasets/lavita/MedQuAD/data/train-00000-of-00001-e36383d177026d53.parquet")
print('Shape:', df.shape)
df.head()

Shape: (47441, 13)


,document_id,document_source,document_url,category,umls_cui,umls_semantic_types,umls_semantic_group,synonyms,question_id,question_focus,question_type,question,answer
0,0000559,GHR,https://ghr.nlm.nih.gov/condition/keratoderma-...,None,C0343073,T047,Disorders,KWWH,0000559-1,keratoderma with woolly hair,information,What is (are) keratoderma with woolly hair ?,Keratoderma with woolly hair is a group of rel...
1,0000559,GHR,https://ghr.nlm.nih.gov/condition/keratoderma-...,None,C0343073,T047,Disorders,KWWH,0000559-2,keratoderma with woolly hair,frequency,How many people are affected by keratoderma wi...,Keratoderma with woolly hair is rare; its prev...
2,0000559,GHR,https://ghr.nlm.nih.gov/condition/keratoderma-...,None,C0343073,T047,Disorders,KWWH,0000559-3,keratoderma with woolly hair,genetic changes,What are the genetic changes related to kerato...,"Mutations in the JUP, DSP, DSC2, and KANK2 gen..."
3,0000559,GHR,https://ghr.nlm.nih.gov/condition/keratoderma-...,None,C0343073,T047,Disorders,KWWH,0000559-4,keratoderma with woolly hair,inheritance,Is keratoderma with woolly hair inherited ?,Most cases of keratoderma with woolly hair hav...
4,0000559,GHR,https://ghr.nlm.nih.gov/condition/keratoderma-...,None,C0343073,T047,Disorders,KWWH,0000559-5,keratoderma with woolly hair,treatment,What are the treatments for keratoderma with w...,These resources address the diagnosis or manag...


### Drop Missing Answers

In [4]:
df.isna().sum()

document_id                5
document_source            0
document_url               0
category               15431
umls_cui               16024
umls_semantic_types    16066
umls_semantic_group    16024
synonyms               22772
question_id                0
question_focus            14
question_type              0
question                   0
answer                 31034
dtype: int64

In [5]:
df.dropna(subset=['answer'], inplace=True)
print('Shape:', df.shape)

Shape: (16407, 13)


### Drop Small question_type Groups

In [6]:
df['question_type'].value_counts()

question_type
information        4535
symptoms           2748
treatment          2442
inheritance        1446
frequency          1120
genetic changes    1087
causes              727
exams and tests     653
research            395
outlook             361
susceptibility      324
considerations      235
prevention          210
stages               77
complications        46
support groups        1
Name: count, dtype: int64

In [7]:
small_groups = ['stages', 'complications', 'support groups', 'prevention', 'considerations']
df = df.loc[~(df['question_type'].isin(small_groups))]
print('Shape:', df.shape)

Shape: (15838, 13)


## Sample and Save 3.5k QAs

In [ ]:
def balanced_group_sample(df, group_cols, n_per_group):
    # Group by specified cols
    grouped = df.groupby(group_cols)
    # Sample n rows (or all, if size is less than n) per group
    balanced_sample = grouped.apply(
        lambda x: x.sample(n=min(n_per_group, len(x)), random_state=42),
        include_groups=False
    ).reset_index()
    return balanced_sample

sampled_df = balanced_group_sample(df, group_cols=["question_type"], n_per_group=324)
print('Shape:', sampled_df.shape)
sampled_df['question_type'].value_counts()
# Save sampled data to CSV
sampled_df.to_csv('../data/medquad_sampled.csv', index=False)
# Save QA only to jsonl
sampled_df = sampled_df[['question', 'answer']]
sampled_df.to_json('../data/medquad_sampled.jsonl', orient='records', lines=True)

Shape: (3564, 14)


---

# 1.2 Prepare for MCQ Generation

In [14]:
sampled_df = pd.read_json('../data/medquad_sampled.jsonl', lines=True)
print('Shape:', sampled_df.shape)
sampled_df.head()

Shape: (3564, 2)


,question,answer
0,What causes Coronary Heart Disease ?,Research suggests that coronary heart disease ...
1,What causes Pyelonephritis: Kidney Infection ?,Pyelonephritis is caused by a bacterium or vir...
2,What causes Type 1 plasminogen deficiency ?,"What causes plasminogen deficiency, type 1? Pl..."
3,What causes Jejunal atresia ?,What causes jejunal atresia? Jejunal atresia o...
4,What causes Erythromelalgia ?,What causes erythromelalgia? About 15% of case...
